## Calling our OpenAI Model from Microsoft Foundry

### Installing Required Libraries

In [1]:
%pip install openai==1.109.1 python-dotenv

Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment Variables

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
load_dotenv(env_path, override=False)

model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o")
connection_string = os.getenv("PROJECT_CONNECTION_STRING")

print("Environment Configuration:")
print(f"  .env file path: {env_path.absolute()}")
print(f"  .env exists: {env_path.exists()}")
print(f"  MODEL_DEPLOYMENT_NAME: {model_deployment_name}")
print()

if not model_deployment_name:
    raise ValueError("❌ Missing MODEL_DEPLOYMENT_NAME in .env file.")
if not connection_string:
    raise ValueError("❌ Missing PROJECT_CONNECTION_STRING in .env file.")

# Derive the Azure OpenAI resource settings from the connection string
conn_parts = dict(part.split("=", 1) for part in connection_string.split(";") if "=" in part)
azure_resource_name = conn_parts["Endpoint"].split("//")[1].split(".")[0]
azure_resource_group = conn_parts["ResourceGroup"]
azure_subscription = conn_parts["SubscriptionId"]

print("Azure OpenAI Resource Settings:")
print(f"  Resource: {azure_resource_name}")
print(f"  Resource Group: {azure_resource_group}")
print()

print("✓ Configuration validated successfully.")

Environment Configuration:
  .env file path: /Users/firozozman/Library/CloudStorage/OneDrive-Personal/Code/MicrosoftFoundry/GettingStarted/OpenAI_Chat_Completions/.env
  .env exists: True
  MODEL_DEPLOYMENT_NAME: gpt-4o

Azure OpenAI Resource Settings:
  Resource: fire-foundry-resource
  Resource Group: foundry-learn-rg

✓ Configuration validated successfully.


### Setting up the Azure OpenAI Client

This cell retrieves the API key and creates an OpenAI client for chatting with your model.

In [4]:
import subprocess
import json
from openai import AzureOpenAI

print("Step 1: Retrieving Azure OpenAI API key...")
try:
    cmd = [
        "az", "cognitiveservices", "account", "keys", "list",
        "--name", azure_resource_name,
        "--resource-group", azure_resource_group,
        "--subscription", azure_subscription
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    keys_data = json.loads(result.stdout)
    api_key = keys_data.get("key1")
    
    if not api_key:
        raise ValueError("No API key found in Azure response")
    
    print(f"  ✓ API key retrieved")
    
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"Failed to retrieve API key: {e.stderr}")
except Exception as e:
    raise RuntimeError(f"Error: {e}")

print()
print("Step 2: Creating Azure OpenAI client...")
try:
    azure_endpoint = f"https://{azure_resource_name}.openai.azure.com/"
    api_version = "2024-08-01-preview"
    
    openai_client = AzureOpenAI(
        api_version=api_version,
        azure_endpoint=azure_endpoint,
        api_key=api_key
    )
    
    print(f"  ✓ Client created successfully")
    print(f"    Endpoint: {azure_endpoint}")
    print(f"    Model: {model_deployment_name}")
    print(f"    API Version: {api_version}")
    
except Exception as e:
    raise RuntimeError(f"Failed to create OpenAI client: {e}")

Step 1: Retrieving Azure OpenAI API key...
  ✓ API key retrieved

Step 2: Creating Azure OpenAI client...
  ✓ Client created successfully
    Endpoint: https://fire-foundry-resource.openai.azure.com/
    Model: gpt-4o
    API Version: 2024-08-01-preview


### Sending a Chat Request

In [5]:
print("Sending request to Azure OpenAI chat.completions...")
print(f"  Model: {model_deployment_name}")
print(f"  System: You are a helpful AI assistant.")
print(f"  User: Can you tell me about Microsoft Foundry?")
print()

try:
    response = openai_client.chat.completions.create(
        model=model_deployment_name,
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": "Can you tell me about Microsoft Foundry?"}
        ]
    )
    
    print("✓ Response received from OpenAI model:")
    print()
    print(response.choices[0].message.content)
    print()
    print(f"Tokens used - Prompt: {response.usage.prompt_tokens}, Completion: {response.usage.completion_tokens}")
except Exception as e:
    print(f"❌ Failed to get response from OpenAI model")
    print(f"  Error: {type(e).__name__}: {str(e)[:300]}")
    raise

Sending request to Azure OpenAI chat.completions...
  Model: gpt-4o
  System: You are a helpful AI assistant.
  User: Can you tell me about Microsoft Foundry?

✓ Response received from OpenAI model:

Absolutely! Microsoft Foundry is an innovation and development initiative within Microsoft that serves as a sandbox environment for creating and prototyping new technologies, fostering creativity, and building experimental solutions that may eventually scale to larger projects. The initiative encourages collaboration across teams and acts as an incubator for novel ideas, focusing on rapidly iterating on software and app designs.

Here are some key aspects of Microsoft Foundry:

1. **Focus on Innovation and Experimentation**:
   Microsoft Foundry is designed to explore new ideas, foster interdisciplinary experimentation, and take creative risks. It allows teams to quickly iterate on potential solutions to solve emerging business or technology challenges.

2. **Internship and Learning Opport